In [1]:
# ============================================================
# IMPORTS ET CHARGEMENT DES DATASETS
import os 
import sys
import pandas as pd
sys.path.append(os.path.abspath("..")) 
from src.models.train_model import preparer_donnees, entrainer_modeles, regression_ridge, valider_modeles_cv
from src.models.predict_model import predire_modele, calculer_metriques_modele
 
dvf_initial = pd.read_csv("../data/processed/dvf_initial.csv", dtype={"code_commune": "string"})
dvf_nettoye = pd.read_csv("../data/processed/dvf_nettoye.csv", dtype={"code_commune": "string"})
dvf_insee = pd.read_csv("../data/processed/dvf_insee_enrichi.csv", dtype={"code_commune": "string"})

print("DVF initial (avec prix extrêmes) :", dvf_initial.shape)
print("DVF nettoyé (sans prix extrêmes) :", dvf_nettoye.shape)
print("DVF enrichi avec les données INSEE :", dvf_insee.shape)
print("Prix extrêmes retirés :", len(dvf_initial) - len(dvf_nettoye))

DVF initial (avec prix extrêmes) : (115106, 16)
DVF nettoyé (sans prix extrêmes) : (110889, 16)
DVF enrichi avec les données INSEE : (110889, 28)
Prix extrêmes retirés : 4217


In [2]:
# ============================================================
# PRÉPARATION DES DONNÉES

X_train_initial, X_test_initial, y_train_initial, y_test_initial = preparer_donnees(dvf_initial)
X_train_nettoye, X_test_nettoye, y_train_nettoye, y_test_nettoye = preparer_donnees(dvf_nettoye)
X_train_insee, X_test_insee, y_train_insee, y_test_insee = preparer_donnees(dvf_insee)

print("Train initial :", X_train_initial.shape)
print("Test initial :", X_test_initial.shape)

print("Train nettoyé :", X_train_nettoye.shape)
print("Test nettoyé :", X_test_nettoye.shape)

print("Train DVF + INSEE :", X_train_insee.shape)
print("Test DVF + INSEE :", X_test_insee.shape)


Train initial : (92084, 14)
Test initial : (23022, 14)
Train nettoyé : (88711, 14)
Test nettoyé : (22178, 14)
Train DVF + INSEE : (88711, 26)
Test DVF + INSEE : (22178, 26)


In [3]:
# ============================================================
# ENTRAÎNEMENT DES MODÈLES DummyRegressor et LinearRegression

dummy_initial, modele_lr_initial = entrainer_modeles(X_train_initial, y_train_initial)
dummy_nettoye, modele_lr_nettoye = entrainer_modeles(X_train_nettoye, y_train_nettoye)
_, modele_lr_insee = entrainer_modeles(X_train_insee, y_train_insee)

print("Les modèles sont entraînés.")

# ============================================================
# PRÉDICTIONS : DummyRegressor et LinearRegression

y_pred_dummy_initial = predire_modele(dummy_initial, X_test_initial)
y_pred_lr_initial = predire_modele(modele_lr_initial, X_test_initial)
y_pred_dummy_nettoye = predire_modele(dummy_nettoye, X_test_nettoye)
y_pred_lr_nettoye = predire_modele(modele_lr_nettoye, X_test_nettoye)
# DVF + INSEE
y_pred_lr_insee = predire_modele(modele_lr_insee, X_test_insee)
print("Les prédictions sont générées.")

# ============================================================
# MÉTRIQUES : DummyRegressor et LinearRegression

resultat_dummy_initial = calculer_metriques_modele(y_test_initial, y_pred_dummy_initial, "DummyRegressor", "Avant traitement des prix extrêmes")
resultat_lr_initial = calculer_metriques_modele(y_test_initial, y_pred_lr_initial, "LinearRegression", "Avant traitement des prix extrêmes")
resultat_dummy_nettoye = calculer_metriques_modele(y_test_nettoye, y_pred_dummy_nettoye, "DummyRegressor", "Après traitement des prix extrêmes")
resultat_lr_nettoye = calculer_metriques_modele(y_test_nettoye, y_pred_lr_nettoye, "LinearRegression", "Après traitement des prix extrêmes")
resultats_lr_insee = calculer_metriques_modele(y_test_insee, y_pred_lr_insee, "Régression linéaire", "DVF + INSEE")

comparaison_baselines = pd.concat([resultat_dummy_initial, resultat_lr_initial, resultat_dummy_nettoye, resultat_lr_nettoye, resultats_lr_insee], ignore_index=True)
display(comparaison_baselines.round(2))


Les modèles sont entraînés.
Les prédictions sont générées.


,Dataset,Modèle,MAE (€/m²),RMSE (€/m²),R²
0,Avant traitement des prix extrêmes,DummyRegressor,1327.15,3291.08,-0.00
1,Avant traitement des prix extrêmes,LinearRegression,962.98,3056.57,0.13
2,Après traitement des prix extrêmes,DummyRegressor,1174.95,1520.96,-0.01
3,Après traitement des prix extrêmes,LinearRegression,815.13,1105.77,0.47
4,DVF + INSEE,Régression linéaire,833.27,1131.89,0.44


### Conclusion de la baseline

La régression linéaire est plus performante que le **DummyRegressor**, ce qui montre que les variables utilisées apportent une information utile pour prédire le prix au m².

Après suppression des prix extrêmes :

la **MAE** passe de **962,98 à 815,13 €/m²** ;
la **RMSE** passe de **3 056,57 à 1 105,77 €/m²** ;
le **R²** passe de **0,13 à 0,47**.

Le traitement des prix extrêmes améliore donc nettement les performances de la régression linéaire.

L'enrichissement avec les données **INSEE** n'améliore pas la baseline linéaire.

À ce stade, les variables INSEE n'apportent donc pas de gain avec une régression linéaire. Leur intérêt sera néanmoins évalué avec des modèles plus complexes, capables de mieux capturer des relations non linéaires et des interactions entre les variables.

In [4]:
# ============================================================
# COMPARAISON SANS ET AVEC FEATURE ENGINEERING

dvf_sans_features = dvf_nettoye.drop(columns=["surface_par_piece", "distance_lyon_km", "log_surface","mois_mutation","annee_mutation"])
X_train_sans_fe, X_test_sans_fe, y_train_sans_fe, y_test_sans_fe = (preparer_donnees(dvf_sans_features))
_, modele_lr_sans_fe = entrainer_modeles(X_train_sans_fe, y_train_sans_fe)

y_pred_lr_sans_fe = predire_modele(modele_lr_sans_fe,   X_test_sans_fe)
resultat_lr_sans_fe = calculer_metriques_modele(y_test_sans_fe, y_pred_lr_sans_fe, "LinearRegression", "Sans features créées")

resultat_lr_avec_fe = resultat_lr_nettoye.copy()
resultat_lr_avec_fe["Dataset"] = "Avec features créées"

comparaison_features = pd.concat([ resultat_lr_sans_fe, resultat_lr_avec_fe], ignore_index=True)
display(comparaison_features.round(2))

,Dataset,Modèle,MAE (€/m²),RMSE (€/m²),R²
0,Sans features créées,LinearRegression,837.78,1133.84,0.44
1,Avec features créées,LinearRegression,815.13,1105.77,0.47


### Évaluation de l’apport du feature engineering

L’ajout des features améliore légèrement les performances de la régression linéaire =>
Ces nouvelles variables apportent donc une information utile au modèle, même si l’amélioration reste modeste. Nous décidons de les conserver pour la suite de la modélisation.

In [5]:
# ============================================================
# Regression RIDGE vs Regression linear

modele_ridge, meilleurs_parametres_ridge, mae_cv_ridge = regression_ridge(X_train_nettoye, y_train_nettoye)

# Réaliser les prédictions + Calculer les métriques du modèle Ridge sur les données de test
y_pred_ridge = predire_modele(modele_ridge, X_test_nettoye)
resultats_ridge = calculer_metriques_modele(y_test_nettoye, y_pred_ridge, "Ridge (régression linéaire régularisée)", "Dataset nettoyé")

# comparer les résultats de la régression linéaire et de Ridge
comparaison_regressions = pd.concat([resultat_lr_nettoye, resultats_ridge], ignore_index=True)
display(comparaison_regressions.round(2))


,Dataset,Modèle,MAE (€/m²),RMSE (€/m²),R²
0,Après traitement des prix extrêmes,LinearRegression,815.13,1105.77,0.47
1,Dataset nettoyé,Ridge (régression linéaire régularisée),814.85,1105.36,0.47


**pour Conclure:** La régression Ridge obtient des résultats presque identiques à ceux de la régression linéaire classique. 
La régularisation n’apporte donc pas d’amélioration. 
Nous conservons la régression linéaire comme baseline et poursuivons la modélisation avec des modèles plus complexes afin d’améliorer les performances prédictives.

In [6]:
# ============================================================
# VALIDATION CROISÉE : RÉGRESSION LINÉAIRE ET RIDGE

modeles_lr_ridge = {
    "Régression linéaire": modele_lr_nettoye,
    "Ridge": modele_ridge
}

resultats_cv_lr_ridge = valider_modeles_cv(modeles=modeles_lr_ridge, X_train=X_train_nettoye, y_train=y_train_nettoye, n_splits=5)
display(resultats_cv_lr_ridge.round(2))

,Modèle,Nombre de folds,MAE moyenne (€/m²),Écart-type (€/m²)
0,Régression linéaire,5,817.57,2.09
1,Ridge,5,817.39,2.14


**Validation croisée de la régression linéaire:** La validation croisée confirme la stabilité de la régression linéaire, avec une MAE moyenne de 817,57 €/m² et un faible écart-type de 2,09 €/m². L’erreur restant élevée, nous testons ensuite un modèle plus complexe : le Random Forest.